# Password Security Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from datetime import datetime
import re

In [ ]:
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## CSV Firefox Export format

"url","username","password","httpRealm","formActionOrigin","guid","timeCreated","timeLastUsed","timePasswordChanged"


In [ ]:
# firefox csv
# "url","username","password","httpRealm","formActionOrigin","guid","timeCreated","timeLastUsed","timePasswordChanged"

CSV_FILE = 'passwords.csv'

df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} password entries")
#WARNING!!!!!!!!!!!!!! DO NOT USE THIS AND GIT COMMIT YOU GONNA LEAK YOUR FIRST FEW LIENS
#df.head()

In [ ]:
UPPER_RE = re.compile(r'[A-Z]')
LOWER_RE = re.compile(r'[a-z]')
NUM_RE = re.compile(r'[0-9]')
SPEC_RE = re.compile(r'[^a-zA-Z0-9]')

def analyze_password_optimized(password):
    if pd.isna(password) or password == '':
        return None

    pwd = str(password)

    return {
        'length': len(pwd),
        'has_upper': bool(UPPER_RE.search(pwd)),
        'has_lower': bool(LOWER_RE.search(pwd)),
        'has_number': bool(NUM_RE.search(pwd)),
        'has_special': bool(SPEC_RE.search(pwd))
    }

analysis = df['password'].apply(analyze_password)
df['length'] = analysis.apply(lambda x: x['length'])
df['has_upper'] = analysis.apply(lambda x: x['has_upper'])
df['has_lower'] = analysis.apply(lambda x: x['has_lower'])
df['has_number'] = analysis.apply(lambda x: x['has_number'])
df['has_special'] = analysis.apply(lambda x: x['has_special'])

print("Password composition analyzed!")

In [ ]:
print("="*60)
print("BASIC STATISTICS")
print("="*60)
print(f"Total passwords: {len(df)}")
print(f"Average length: {df['length'].mean():.2f} characters")
print(f"Median length: {df['length'].median():.0f} characters")
print(f"Min length: {df['length'].min()}")
print(f"Max length: {df['length'].max()}")

In [ ]:
total = len(df)
print("="*60)
print("PASSWORD COMPOSITION")
print("="*60)
print(f"Contain UPPERCASE: {df['has_upper'].sum()} ({df['has_upper'].sum()/total*100:.1f}%)")
print(f"Contain lowercase: {df['has_lower'].sum()} ({df['has_lower'].sum()/total*100:.1f}%)")
print(f"Contain NUMBERS: {df['has_number'].sum()} ({df['has_number'].sum()/total*100:.1f}%)")
print(f"Contain SPECIAL chars: {df['has_special'].sum()} ({df['has_special'].sum()/total*100:.1f}%)")

# All four types
all_four = df[(df['has_upper']) & (df['has_lower']) & (df['has_number']) & (df['has_special'])]
print(f"\nAll 4 types (Upper+Lower+Number+Special): {len(all_four)} ({len(all_four)/total*100:.1f}%)")

In [ ]:
# Character Type Distribution
fig, ax = plt.subplots(figsize=(8, 6))
composition = {
    'Uppercase': df['has_upper'].sum(),
    'Lowercase': df['has_lower'].sum(),
    'Numbers': df['has_number'].sum(),
    'Special': df['has_special'].sum()
}
bars = ax.bar(composition.keys(), composition.values(), color=['black', 'black', 'black', 'black'], edgecolor='black')
ax.set_ylabel('Number of Passwords')
ax.set_title('Distribution based on Character Type')
for bar in bars:
    ax.annotate(f'{int(bar.get_height())}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['created_date'].dropna().hist(ax=axes[0], bins=20, edgecolor='black', color='black')
axes[0].set_title('Password Creation Timeline')
axes[0].set_xlabel('Date')
axes[0].tick_params(axis='x', rotation=45)

df['changed_date'].dropna().hist(ax=axes[1], bins=20, edgecolor='black', color='black')
axes[1].set_title('Password Changed Timeline')
axes[1].set_xlabel('Date')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
def get_strength(row):
    score = 0
    if row['length'] >= 8: score += 1
    if row['length'] >= 12: score += 1
    if row['has_upper']: score += 1
    if row['has_lower']: score += 1
    if row['has_number']: score += 1
    if row['has_special']: score += 1
    if score <= 2: return 'Weak'
    elif score <= 4: return 'Medium'
    return 'Strong'

df['strength'] = df.apply(get_strength, axis=1)

fig, ax = plt.subplots(figsize=(8, 6))
strength_counts = df['strength'].value_counts()
colors = {'Weak': 'black', 'Medium': 'black', 'Strong': 'black'}
bars = ax.bar(strength_counts.index, strength_counts.values,
              color=[colors.get(x, 'gray') for x in strength_counts.index], edgecolor='black')
ax.set_ylabel('Count')
ax.set_title('Password Strength Distribution')
for bar in bars:
    ax.annotate(f'{int(bar.get_height())}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom')
plt.show()

print("\nStrength Distribution:")
for s in ['Weak', 'Medium', 'Strong']:
    c = strength_counts.get(s, 0)
    print(f"{s}: {c} ({c/len(df)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full range with log scale
axes[0].hist(df['length'], bins=50, edgecolor='black', color='black', alpha=0.7)
axes[0].axvline(df['length'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean:{df["length"].mean():.1f}')
axes[0].set_yscale('log')
axes[0].set_xlabel('Password Length')
axes[0].set_ylabel('Frequency (log)')
axes[0].set_title('Distribution (log scale)')
axes[0].legend()

# Zoomed 0-30 range
axes[1].hist(df['length'], bins=range(0, 32), edgecolor='black', color='black', alpha=0.7)
axes[1].axvline(df['length'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean:{df["length"].mean():.1f}')
axes[1].set_xlabel('Password Length')
axes[1].set_ylabel('Frequency')
axes[1].set_title('0-30 chars')
axes[1].set_xlim(0, 30)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Password Length Statistics
fig, ax = plt.subplots(figsize=(8, 5))

categories =['Average', 'Median', 'Min', 'Max']

values = [
    round(df['length'].mean(), 2),
    int(df['length'].median()),
    df['length'].min(),
    df['length'].max()
]

colors = ['black', 'black', 'black', 'black']

bars = ax.barh(categories, values, color=colors, edgecolor='black', height=0.6)
ax.set_xlabel('Characters')
ax.set_title('Password Length Statistics', fontsize=14, fontweight='bold')

for bar, val in zip(bars, values):
      ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val}',
              va='center', fontsize=12, fontweight='bold')

# 2. DYNAMICALLY SCALE THE GRAPH (instead of hardcoding 350)
ax.set_xlim(0, df['length'].max() * 1.15)

plt.tight_layout()
plt.savefig('pw_count_stats.png', dpi=150, bbox_inches='tight')
plt.show()